# Hospital AI Appointment Agent - MediLLM

This notebook contains your MediLLM appointment agent code and the provided output below it.

**Security note:** the exposed Groq API key has been replaced with a placeholder in this notebook.


In [ ]:
from openai import OpenAI
from typing import Dict, Any, Optional
from datetime import datetime, timedelta
import re
import json
import random

# Initialize Groq client
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key="YOUR_GROQ_API_KEY_HERE"
)

class AppointmentTools:
    """Tool executor for appointment actions"""

    def __init__(self):
        # Mock appointment database
        self.appointments_db = {
            "dr_sharma": {
                "2026-03-19": ["09:00", "10:00", "11:00", "14:00", "15:00", "16:00"],
                "2026-03-20": ["09:00", "10:00", "11:00", "14:00", "15:00", "16:00"],
                "2026-03-21": ["09:00", "11:00", "14:00", "15:00", "16:00"]  # 10 AM booked
            }
        }

    def check_availability(self, doctor: str, date: str, time: str) -> bool:
        """Check if slot is available"""
        doctor_key = doctor.lower().replace(" ", "_").replace("dr.", "dr")

        if doctor_key in self.appointments_db:
            if date in self.appointments_db[doctor_key]:
                return time in self.appointments_db[doctor_key][date]
        return False

    def confirm_slot(self, doctor: str, date: str, time: str, patient: str = "Patient") -> str:
        """Tool 1: Confirm appointment slot"""
        confirmation_id = f"APT{random.randint(10000, 99999)}"

        return f"""
╔════════════════════════════════════════════════════════════════╗
║                    ✅ APPOINTMENT CONFIRMED                     ║
╠════════════════════════════════════════════════════════════════╣
║  Confirmation ID: {confirmation_id}                                         
║  Patient: {patient}                                              
║  Doctor: Dr. {doctor}                                            
║  Date: {date}                                                    
║  Time: {time}                                                    
║  Status: ✅ Confirmed                                            
║  Instructions: Please arrive 15 minutes early                  
╚════════════════════════════════════════════════════════════════╝
"""

    def suggest_reschedule(self, doctor: str, date: str, time: str, patient: str = "Patient") -> str:
        """Tool 2: Suggest rescheduling options"""
        doctor_key = doctor.lower().replace(" ", "_").replace("dr.", "dr")
        available_slots = []

        if doctor_key in self.appointments_db and date in self.appointments_db[doctor_key]:
            all_slots = ["09:00", "10:00", "11:00", "14:00", "15:00", "16:00"]
            booked = self.appointments_db[doctor_key][date]
            available_slots = [slot for slot in all_slots if slot not in booked]

        alternatives = ", ".join(available_slots[:3]) if available_slots else "tomorrow"

        return f"""
╔════════════════════════════════════════════════════════════════╗
║                    ⚠️ SLOT UNAVAILABLE                          ║
╠════════════════════════════════════════════════════════════════╣
║  Patient: {patient}                                              
║  Doctor: Dr. {doctor}                                            
║  Date: {date}                                                    
║  Requested Time: {time} - ❌ NOT AVAILABLE                       
║                                                                  
║  📅 Available alternatives today:                                
║  {alternatives}                                                  
║                                                                  
║  Please contact reception to reschedule: 555-MEDICAL           
╚════════════════════════════════════════════════════════════════╝
"""

    def suggest_alternative_doctor(self, doctor: str, date: str, time: str, patient: str = "Patient") -> str:
        """Tool 3: Suggest alternative doctor"""
        return f"""
╔════════════════════════════════════════════════════════════════╗
║                    👨‍⚕️ ALTERNATIVE DOCTOR                        ║
╠════════════════════════════════════════════════════════════════╣
║  Dr. {doctor} is fully booked on {date}                          
║                                                                  
║  Available doctors:                                              
║  • Dr. Patel - {time} available                                  
║  • Dr. Singh - {time} available                                  
║  • Dr. Kumar - {time} available                                  
║                                                                  
║  Would you like to book with an alternative?                    
╚════════════════════════════════════════════════════════════════╝
"""

class MediLLM:
    """Hospital AI Appointment Agent"""

    def __init__(self):
        self.tools = AppointmentTools()
        self.patient_context = {}

    def parse_request(self, request: str) -> Dict[str, str]:
        """Extract doctor, date, time from request"""
        request_lower = request.lower()

        doctor_match = re.search(r'dr\.?\s*(\w+)', request_lower)
        doctor = doctor_match.group(1).title() if doctor_match else "Sharma"

        time_match = re.search(r'(\d{1,2})\s*(?::| )?(\d{2})?\s*(am|pm)?', request_lower)
        if time_match:
            hour = time_match.group(1)
            minute = time_match.group(2) if time_match.group(2) else "00"
            meridiem = time_match.group(3) if time_match.group(3) else "am"
            time = f"{hour}:{minute} {meridiem.upper()}"
        else:
            time = "10:00 AM"

        if "tomorrow" in request_lower:
            date = (datetime.now() + timedelta(days=1)).strftime("%Y-%m-%d")
        elif "today" in request_lower:
            date = datetime.now().strftime("%Y-%m-%d")
        else:
            date = (datetime.now() + timedelta(days=1)).strftime("%Y-%m-%d")

        patient_match = re.search(r'for\s+([a-z\s]+?)(?:\s+at|\s+with|\s+on|$)', request_lower)
        patient = patient_match.group(1).strip().title() if patient_match else "Patient"

        return {
            "doctor": doctor,
            "date": date,
            "time": time,
            "patient": patient,
            "raw_request": request
        }

    def step1_think(self, parsed_request: Dict[str, str]) -> str:
        thought = f"""
🧠 STEP 1: THINKING (Analyzing Request)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🤔 MediLLM analyzing: 
   Patient: {parsed_request['patient']}
   Doctor: Dr. {parsed_request['doctor']}
   Date: {parsed_request['date']}
   Time: {parsed_request['time']}

📋 Checking availability in hospital system...
"""
        print(thought)
        return thought

    def step2_plan(self, parsed_request: Dict[str, str]) -> str:
        is_available = self.tools.check_availability(
            parsed_request['doctor'],
            parsed_request['date'],
            parsed_request['time'].split()[0]
        )

        if is_available:
            plan = f"""
📋 STEP 2: PLANNING (Choosing Action)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ Slot is AVAILABLE!
   Action Selected: confirm_slot
   Tool: AppointmentTools.confirm_slot()
   Parameters: doctor={parsed_request['doctor']}, date={parsed_request['date']}, 
               time={parsed_request['time']}, patient={parsed_request['patient']}
"""
            self.selected_action = "confirm"
        else:
            has_alternatives = random.choice([True, False])

            if has_alternatives:
                plan = f"""
📋 STEP 2: PLANNING (Choosing Action)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
❌ Slot {parsed_request['time']} is BOOKED!
   ✅ Alternative slots available today
   Action Selected: suggest_reschedule
   Tool: AppointmentTools.suggest_reschedule()
   Parameters: doctor={parsed_request['doctor']}, date={parsed_request['date']}, 
               time={parsed_request['time']}, patient={parsed_request['patient']}
"""
                self.selected_action = "reschedule"
            else:
                plan = f"""
📋 STEP 2: PLANNING (Choosing Action)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
❌ Dr. {parsed_request['doctor']} fully booked on {parsed_request['date']}
   ✅ Alternative doctors available
   Action Selected: suggest_alternative_doctor
   Tool: AppointmentTools.suggest_alternative_doctor()
   Parameters: doctor={parsed_request['doctor']}, date={parsed_request['date']}, 
               time={parsed_request['time']}, patient={parsed_request['patient']}
"""
                self.selected_action = "alternative"

        print(plan)
        return plan

    def step3_act(self, parsed_request: Dict[str, str]) -> str:
        print("\n⚙️ STEP 3: ACTING (Running Tools)")
        print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

        if self.selected_action == "confirm":
            result = self.tools.confirm_slot(
                parsed_request['doctor'],
                parsed_request['date'],
                parsed_request['time'].split()[0],
                parsed_request['patient']
            )
        elif self.selected_action == "reschedule":
            result = self.tools.suggest_reschedule(
                parsed_request['doctor'],
                parsed_request['date'],
                parsed_request['time'].split()[0],
                parsed_request['patient']
            )
        else:
            result = self.tools.suggest_alternative_doctor(
                parsed_request['doctor'],
                parsed_request['date'],
                parsed_request['time'].split()[0],
                parsed_request['patient']
            )

        print(result)
        return result

    def step4_answer(self, action_result: str) -> str:
        print("\n🎯 STEP 4: ANSWER (Final Decision)")
        print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

        if "CONFIRMED" in action_result:
            decision = "Decision: ✅ APPOINTMENT CONFIRMED"
        elif "UNAVAILABLE" in action_result:
            decision = "Decision: ⚠️ SLOT UNAVAILABLE - PLEASE RESCHEDULE"
        else:
            decision = "Decision: 👨‍⚕️ ALTERNATIVE DOCTOR SUGGESTED"

        final = f"""
🏥 MEDILLM FINAL DECISION:
{decision}

{action_result}
"""
        print(final)
        return final

    def process_request(self, request: str) -> str:
        print("\n" + "=" * 70)
        print("🏥 HOSPITAL AI APPOINTMENT AGENT - MEDILLM".center(70))
        print("=" * 70)

        parsed = self.parse_request(request)
        self.step1_think(parsed)
        self.step2_plan(parsed)
        action_result = self.step3_act(parsed)
        final_decision = self.step4_answer(action_result)

        return final_decision

def patient_request_simulator():
    medi = MediLLM()

    test_requests = [
        "Request for Dr. Sharma at 10 AM tomorrow for John",
        "Need appointment with Dr. Patel at 2 PM today for Sarah",
        "Book Dr. Sharma at 11 AM tomorrow for Michael",
        "Emergency appointment with Dr. Kumar at 3 PM for baby Emma",
        "Follow-up with Dr. Sharma at 10 AM today for Robert"
    ]

    for i, request in enumerate(test_requests, 1):
        print(f"\n{'#' * 70}")
        print(f"PATIENT REQUEST #{i}".center(70))
        print(f"Request: '{request}'")
        print(f"{'#' * 70}")

        medi.process_request(request)

        if i < len(test_requests):
            input("\nPress Enter for next patient request...")

if __name__ == "__main__":
    medi = MediLLM()
    result = medi.process_request("Request for Dr. Sharma at 10 AM tomorrow for Amit Kumar")
    print(result)


## Output

In [1]:
print(r"""======================================================================
              🏥 HOSPITAL AI APPOINTMENT AGENT - MEDILLM               
======================================================================

🧠 STEP 1: THINKING (Analyzing Request)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🤔 MediLLM analyzing: 
   Patient: Amit Kumar
   Doctor: Dr. Sharma
   Date: 2026-03-19
   Time: 10:00 AM

📋 Checking availability in hospital system...


📋 STEP 2: PLANNING (Choosing Action)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
❌ Slot 10:00 AM is BOOKED!
   ✅ Alternative slots available today
   Action Selected: suggest_reschedule
   Tool: AppointmentTools.suggest_reschedule()
   Parameters: doctor=Sharma, date=2026-03-19, 
               time=10:00 AM, patient=Amit Kumar


⚙️ STEP 3: ACTING (Running Tools)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╔════════════════════════════════════════════════════════════════╗
║                    ⚠️ SLOT UNAVAILABLE                          ║
╠════════════════════════════════════════════════════════════════╣
║  Patient: Amit Kumar                                              
║  Doctor: Dr. Sharma                                            
║  Date: 2026-03-19                                                    
║  Requested Time: 10:00 - ❌ NOT AVAILABLE                       
║                                                                  
║  📅 Available alternatives today:                                
║  tomorrow                                                  
║                                                                  
║  Please contact reception to reschedule: 555-MEDICAL           
╚════════════════════════════════════════════════════════════════╝


🎯 STEP 4: ANSWER (Final Decision)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🏥 MEDILLM FINAL DECISION:
Decision: ⚠️ SLOT UNAVAILABLE - PLEASE RESCHEDULE


╔════════════════════════════════════════════════════════════════╗
║                    ⚠️ SLOT UNAVAILABLE                          ║
╠════════════════════════════════════════════════════════════════╣
║  Patient: Amit Kumar                                              
║  Doctor: Dr. Sharma                                            
║  Date: 2026-03-19                                                    
║  Requested Time: 10:00 - ❌ NOT AVAILABLE                       
║                                                                  
║  📅 Available alternatives today:                                
║  tomorrow                                                  
║                                                                  
║  Please contact reception to reschedule: 555-MEDICAL           
╚════════════════════════════════════════════════════════════════╝



🏥 MEDILLM FINAL DECISION:
Decision: ⚠️ SLOT UNAVAILABLE - PLEASE RESCHEDULE


╔════════════════════════════════════════════════════════════════╗
║                    ⚠️ SLOT UNAVAILABLE                          ║
╠════════════════════════════════════════════════════════════════╣
║  Patient: Amit Kumar                                              
║  Doctor: Dr. Sharma                                            
║  Date: 2026-03-19                                                    
║  Requested Time: 10:00 - ❌ NOT AVAILABLE                       
║                                                                  
║  📅 Available alternatives today:                                
║  tomorrow                                                  
║                                                                  
║  Please contact reception to reschedule: 555-MEDICAL           
╚════════════════════════════════════════════════════════════════╝
""")

              🏥 HOSPITAL AI APPOINTMENT AGENT - MEDILLM               

🧠 STEP 1: THINKING (Analyzing Request)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🤔 MediLLM analyzing: 
   Patient: Amit Kumar
   Doctor: Dr. Sharma
   Date: 2026-03-19
   Time: 10:00 AM

📋 Checking availability in hospital system...


📋 STEP 2: PLANNING (Choosing Action)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
❌ Slot 10:00 AM is BOOKED!
   ✅ Alternative slots available today
   Action Selected: suggest_reschedule
   Tool: AppointmentTools.suggest_reschedule()
   Parameters: doctor=Sharma, date=2026-03-19, 
               time=10:00 AM, patient=Amit Kumar


⚙️ STEP 3: ACTING (Running Tools)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╔════════════════════════════════════════════════════════════════╗
║                    ⚠️ SLOT UNAVAILABLE                          ║
╠════════════════════════════════════════════════════════════════╣
║  Patient: Amit Kumar                                              
║  Doctor: Dr. Sha